In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [2]:
df=pd.read_csv('Churn_Modelling.csv')

In [3]:
df.head(5)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
#data preprocessing
#remove irrelevent columns
df=df.drop(['RowNumber','CustomerId','Surname'],axis=True)

In [5]:
## encode categorical data
label_encoder=LabelEncoder()
df['Gender']=label_encoder.fit_transform(df['Gender'])

In [6]:
df.head(5)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [7]:
## One hot encoding 
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder=OneHotEncoder()
geo_encoder=one_hot_encoder.fit_transform(df[['Geography']])

In [8]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [9]:
one_hot_encoder.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [10]:
geo_encoder_df = pd.DataFrame(
    geo_encoder.toarray(),
    columns=one_hot_encoder.get_feature_names_out(['Geography'])
)


In [11]:
geo_encoder_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [12]:
# combine all the columns with the original data
df=pd.concat([df.drop('Geography',axis=1),geo_encoder_df],axis=1)

In [13]:
df.head(5)

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [14]:
with open('regression_label_encoder.pkl', 'wb') as file:
    pickle.dump(label_encoder,file)

with open('regression_one_hot_encoder.pkl', 'wb') as file:
    pickle.dump(one_hot_encoder,file)


In [15]:
## Divide the dataset into dependent and independent
X=df.drop('EstimatedSalary',axis=1)
y=df['EstimatedSalary']

In [16]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
##scale the features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [17]:
with open('regression_scaler.pkl','wb') as file:
    pickle.dump(scaler,file)


## ANN Regression Problem statement

In [18]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [19]:
## Verify Apple Silicon GPU (Metal) is available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU available: {gpus[0].name}')
else:
    print('No GPU found — running on CPU. Install tensorflow-metal for Apple Silicon acceleration.')


GPU available: /physical_device:GPU:0


In [20]:
## Build ANN model
model=Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    Dense(64,activation='relu'),
    Dense(32,activation='relu'),
    Dense(1)  # Linear activation (default) for regression output
])

## compile the model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])


2026-06-07 01:04:14.204168: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5
2026-06-07 01:04:14.204187: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-06-07 01:04:14.204192: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
I0000 00:00:1780774454.204202  778419 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1780774454.204217  778419 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [21]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
## Setup the tensorboard
log_dir='regressionlogs/fit/'+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [23]:
## setup Early stopping
early_stopping_callbacks=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [24]:
## train the model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    batch_size=256,
    callbacks=[tensorflow_callback,early_stopping_callbacks]
)


Epoch 1/100


2026-06-07 01:04:14.943666: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 100431.0391 - mae: 100431.0391 - val_loss: 98725.4531 - val_mae: 98725.4531
Epoch 2/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 100430.3125 - mae: 100430.3125 - val_loss: 98724.4922 - val_mae: 98724.4922
Epoch 3/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 100429.0312 - mae: 100429.0312 - val_loss: 98722.8594 - val_mae: 98722.8594
Epoch 4/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 100426.7891 - mae: 100426.7891 - val_loss: 98719.8906 - val_mae: 98719.8906
Epoch 5/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 100422.9922 - mae: 100422.9922 - val_loss: 98715.0859 - val_mae: 98715.0859
Epoch 6/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 100417.0234 - mae: 100417.0234 - val_loss: 98708.3750 - val_mae: 98708.3750
Epoch 7/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 100408.9922 - mae: 100408.9922 - val_loss: 98699.0547 - val_mae: 98699.0547
Epoch 8/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 100398.

In [25]:
model.save('regression_model.keras')


In [26]:
%load_ext tensorboard

In [27]:
%tensorboard --logdir regressionlogs/fit